# Music Generation with VAE

## 1.1 Dependencies

In [ ]:
# Import Tensorflow 2.0
#%tensorflow_version 2.x
import tensorflow as tf

# Download and import the MIT 6.S191 package
!pip install mitdeeplearning
import mitdeeplearning as mdl

# Import all remaining packages
import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
!apt-get install abcmidi timidity > /dev/null 2>&1

## 1.2 Dataset

In [ ]:
# Download the dataset
songs = mdl.lab1.load_training_data()

# Print one of the songs to inspect it in greater detail!
example_song = songs[0]
print("\nExample song: ")
print(example_song)

In [ ]:
# Convert the ABC notation to audio file and listen to it
mdl.lab1.play_song(example_song)

In [ ]:
# Join our list of song strings into a single string containing all songs
songs_joined = "\n\n".join(songs)

# Find all unique characters in the joined string
vocab = sorted(set(songs_joined))
print("There are", len(vocab), "unique characters in the dataset")

## 1.3 Process the dataset for the learning task

### Vectorize the text

In [ ]:
### Define numerical representation of text ###

# Create a mapping from character to unique index.
# For example, to get the index of the character "d",
#   we can evaluate `char2idx["d"]`.
char2idx = {u:i for i, u in enumerate(vocab)}

# Create a mapping from indices to characters. This is
#   the inverse of char2idx and allows us to convert back
#   from unique index to the character in our vocabulary.
idx2char = np.array(vocab)

In [ ]:
print('{')
for char,_ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

In [ ]:
### Vectorize the songs string ###

'''TODO: Write a function to convert the all songs string to a vectorized
    (i.e., numeric) representation. Use the appropriate mapping
    above to convert from vocab characters to the corresponding indices.

  NOTE: the output of the `vectorize_string` function
  should be a np.array with `N` elements, where `N` is
  the number of characters in the input string
'''

def vectorize_string(string):
  vectorized_songs = np.array([char2idx[song] for song in string ])
  return vectorized_songs
vectorized_songs = vectorize_string(songs_joined)

In [ ]:
print ('{} ---- characters mapped to int ----> {}'.format(repr(songs_joined[:10]), vectorized_songs[:10]))
# check that vectorized_songs is a numpy array
assert isinstance(vectorized_songs, np.ndarray), "returned result should be a numpy array"

### Create training examples and targets

In [ ]:
### Batch definition to create training examples ###

def get_batch(vectorized_songs, seq_length, batch_size):
  # the length of the vectorized songs string
  n = vectorized_songs.shape[0] - 1
  # randomly choose the starting indices for the examples in the training batch
  idx = np.random.choice(n-seq_length, batch_size)

  '''TODO: construct a list of input sequences for the training batch'''
  input_batch = [vectorized_songs[i:i+seq_length] for i in idx]
  '''TODO: construct a list of output sequences for the training batch'''
  output_batch = [vectorized_songs[i+1:i+seq_length+1] for i in idx]

  # x_batch, y_batch provide the true inputs and targets for network training
  x_batch = np.reshape(input_batch, [batch_size, seq_length])
  y_batch = np.reshape(output_batch, [batch_size, seq_length])
  return x_batch, y_batch


# Perform some simple tests to make sure your batch function is working properly!
test_args = (vectorized_songs, 10, 2)
if not mdl.lab1.test_batch_func_types(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_shapes(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_next_step(get_batch, test_args):
   print("======\n[FAIL] could not pass tests")
else:
   print("======\n[PASS] passed all tests!")

In [ ]:
x_batch, y_batch = get_batch(vectorized_songs, seq_length=5, batch_size=1)

for i, (input_idx, target_idx) in enumerate(zip(np.squeeze(x_batch), np.squeeze(y_batch))):
    print("Step {:3d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx])))

## 1.4 Define the Variational AutoEncoders (VAE) model

In [ ]:
class MusicVAE(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, latent_dim, seq_length):
        super(MusicVAE, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.latent_dim = latent_dim
        self.seq_length = seq_length

        # ===== Simplified Encoder =====
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

        # Let TF use the fast cuDNN GRU on GPU
        self.encoder_gru = tf.keras.layers.GRU(
            64,
            return_sequences=False,
            reset_after=True,   # IMPORTANT for cuDNN
            # remove implementation, let TF decide
        )

        self.encoder_dense = tf.keras.layers.Dense(128, activation='relu')
        self.encoder_dropout = tf.keras.layers.Dropout(0.2)
        self.z_mean_layer = tf.keras.layers.Dense(latent_dim)
        self.z_log_var_layer = tf.keras.layers.Dense(latent_dim)

        # ===== Simplified Decoder =====
        self.decoder_dense1 = tf.keras.layers.Dense(128, activation='relu')
        self.decoder_dense2 = tf.keras.layers.Dense(seq_length * 32, activation='relu')
        self.decoder_reshape = tf.keras.layers.Reshape((seq_length, 32))

        self.decoder_gru = tf.keras.layers.GRU(
            64,
            return_sequences=True,
            reset_after=True,   # IMPORTANT
        )

        self.output_dense = tf.keras.layers.Dense(vocab_size, activation='softmax')

    @staticmethod
    def sampling(args):
        z_mean, z_log_var = args
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    def encode(self, x):
        x = self.embedding(x)
        x = self.encoder_gru(x)
        x = self.encoder_dense(x)
        x = self.encoder_dropout(x)
        z_mean = self.z_mean_layer(x)
        z_log_var = self.z_log_var_layer(x)
        return z_mean, z_log_var

    def decode(self, z):
        x = self.decoder_dense1(z)
        x = self.decoder_dense2(x)
        x = self.decoder_reshape(x)
        x = self.decoder_gru(x)
        return self.output_dense(x)

    def call(self, inputs, training=None):
        z_mean, z_log_var = self.encode(inputs)
        z = self.sampling([z_mean, z_log_var])
        reconstructed = self.decode(z)
        return reconstructed, z_mean, z_log_var

    def generate(self, num_samples=1, temperature=1.0):
        z = tf.random.normal(shape=(num_samples, self.latent_dim))
        generated = self.decode(z)
        if temperature != 1.0:
            generated = generated / temperature
            generated = tf.nn.softmax(generated, axis=-1)
        return generated

### Test out the VAE model

### Predictions from the untrained model

## 1.5 Training the model: loss and training operations

In [ ]:
### Highly optimized hyperparameters for fast CPU training ###
num_training_iterations = 50000
batch_size = 64  # Increased for better throughput
seq_length = 100   # Reduced sequence length for speed (was 100)
learning_rate = 1e-3  # Higher learning rate for faster convergence

vocab_size = len(vocab)
embedding_dim = 256   # Drastically reduced (was 64)
latent_dim = 64     # Reduced (was 128)
beta = 0.1  # Lower beta for faster training

checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "fast_cpu_vae")

model = MusicVAE(vocab_size, embedding_dim, latent_dim, seq_length)
optimizer = tf.keras.optimizers.Adam(learning_rate, clipnorm=0.5)  # Lower clip for speed

@tf.function
def compute_vae_loss(y_true, y_pred, z_mean, z_log_var, beta=0.1):
    recon_loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=False)
    recon_loss = tf.reduce_mean(recon_loss)

    kl_loss = -0.5 * tf.reduce_mean(
        tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
    )

    return recon_loss + beta * kl_loss

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        y_hat, z_mean, z_log_var = model(x, training=True)
        loss = compute_vae_loss(y, y_hat, z_mean, z_log_var, beta)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

##################
# Begin training!#
##################

# ---- ULTRA-FAST CPU DATASET PIPELINE ----
def create_dataset(data, seq_length=64, batch_size=64):
    ds = tf.data.Dataset.from_tensor_slices(data)
    ds = ds.window(seq_length + 1, shift=seq_length, drop_remainder=True)  # No overlap for speed
    ds = ds.flat_map(lambda w: w.batch(seq_length + 1))
    ds = ds.map(lambda seq: (seq[:-1], seq[1:]), num_parallel_calls=2)  # Limited parallelism
    return ds.batch(batch_size).prefetch(1)  # Minimal prefetch

train_dataset = create_dataset(vectorized_songs, seq_length, batch_size)

# Create infinite dataset without using .repeat()
def infinite_dataset_generator():
    while True:
        for batch in train_dataset:
            yield batch

train_iter = infinite_dataset_generator()

history = []
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')
if hasattr(tqdm, '_instances'): tqdm._instances.clear()

os.makedirs(checkpoint_dir, exist_ok=True)

for iter in tqdm(range(num_training_iterations)):
    # Get batch from dataset iterator (FAST)
    x_batch, y_batch = next(train_iter)
    loss = train_step(x_batch, y_batch)

    history.append(loss.numpy())
    plotter.plot(history)

    if iter % 1000 == 0:  # Save even less frequently (was 500)
        model.save_weights(checkpoint_prefix + '.weights.h5')

# Save final model
model.save_weights(checkpoint_prefix + '.weights.h5')

In [ ]:
def calculate_accuracy(y_true, y_pred):
    """Calculate accuracy based on token IDs for VAE with potentially different sequence lengths."""
    y_pred_ids = tf.argmax(y_pred, axis=-1, output_type=tf.int32)

    # Handle shape mismatch by taking minimum length
    min_length = tf.minimum(tf.shape(y_true)[1], tf.shape(y_pred_ids)[1])

    # Slice both to same length
    y_true_sliced = y_true[:, :min_length]
    y_pred_sliced = y_pred_ids[:, :min_length]

    correct_predictions = tf.equal(tf.cast(y_true_sliced, tf.int32), y_pred_sliced)
    accuracy = tf.reduce_mean(tf.cast(correct_predictions, tf.float32))
    return accuracy.numpy()

def evaluate_accuracy(model, dataset):
    """Evaluate accuracy of the VAE model on given dataset."""
    accuracies = []
    for x_batch, y_batch in dataset:
        y_pred, _, _ = model(x_batch, training=False)  # unpack VAE returns tuple
        accuracy = calculate_accuracy(y_batch, y_pred)
        accuracies.append(accuracy)
    mean_accuracy = np.mean(accuracies)
    print(f"VAE Reconstruction Accuracy: {mean_accuracy:.4f}")
    return mean_accuracy

# Example usage with proper VAE evaluation:
x_val, y_val = get_batch(vectorized_songs, seq_length=100, batch_size=32)
val_dataset = [(x_val, y_val)]
mean_accuracy = evaluate_accuracy(model, val_dataset)

**Calculate Note Accuracy, Chord Accuracy and Perplexity**

In [ ]:
import tensorflow as tf
import numpy as np

# NOTE ACCURACY: token-level accuracy with shape handling
def calculate_accuracy(y_true, y_pred):
    """Calculate accuracy comparing integer tokens to predicted argmax with shape handling."""
    y_pred_ids = tf.argmax(y_pred, axis=-1, output_type=tf.int32)

    # Handle shape mismatch by taking minimum length
    min_length = tf.minimum(tf.shape(y_true)[1], tf.shape(y_pred_ids)[1])

    # Slice both to same length
    y_true_sliced = y_true[:, :min_length]
    y_pred_sliced = y_pred_ids[:, :min_length]

    correct_predictions = tf.equal(tf.cast(y_true_sliced, tf.int32), y_pred_sliced)
    accuracy = tf.reduce_mean(tf.cast(correct_predictions, tf.float32))
    return accuracy.numpy()

# CROSS-ENTROPY LOSS (sparse, with shape handling)
def compute_loss(y_true, y_pred):
    """Compute loss with shape handling for VAE models."""
    # Handle shape mismatch
    min_length = tf.minimum(tf.shape(y_true)[1], tf.shape(y_pred)[1])

    # Slice both to same length
    y_true_sliced = y_true[:, :min_length]
    y_pred_sliced = y_pred[:, :min_length, :]

    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
    return loss_fn(y_true_sliced, y_pred_sliced).numpy()

# PERPLEXITY = exp(loss) with shape handling
def calculate_perplexity(y_true, y_pred):
    """Calculate perplexity with shape handling."""
    # Handle shape mismatch
    min_length = tf.minimum(tf.shape(y_true)[1], tf.shape(y_pred)[1])

    # Slice both to same length
    y_true_sliced = y_true[:, :min_length]
    y_pred_sliced = y_pred[:, :min_length, :]

    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
    loss = loss_fn(y_true_sliced, y_pred_sliced)
    return tf.exp(loss).numpy()

# FULL DATASET EVALUATION
def evaluate_full_dataset(model, dataset, is_vae=True):
    total_loss = total_acc = total_perplex = 0.0
    num_batches = 0

    for x_batch, y_batch in dataset:
        if is_vae:
            y_pred, _, _ = model(x_batch, training=False)  # VAE returns tuple
        else:
            y_pred = model(x_batch, training=False)        # RNN returns only y_pred

        loss = compute_loss(y_batch, y_pred)
        acc = calculate_accuracy(y_batch, y_pred)
        perplexity = calculate_perplexity(y_batch, y_pred)

        total_loss += loss
        total_acc += acc
        total_perplex += perplexity
        num_batches += 1

    avg_loss = total_loss / num_batches
    avg_acc = total_acc / num_batches
    avg_perplex = total_perplex / num_batches

    print("\n VAE Evaluation Results (Full Dataset):")
    print(f"- Avg Loss:      {avg_loss:.4f}")
    print(f"- Note Accuracy: {avg_acc:.4f} (reconstruction quality)")
    print(f"- Perplexity:    {avg_perplex:.4f}")
    print(f"- Shape Info:    Input seq_len vs Output seq_len handled automatically")

    return avg_acc, avg_perplex

# Example usage for full dataset
val_dataset = create_dataset(vectorized_songs, seq_length=100, batch_size=64)  # use your dataset pipeline
avg_note_acc, avg_perplex = evaluate_full_dataset(model, val_dataset, is_vae=True)

## 1.7 Generate music using the VAE model

### Restore the latest checkpoint

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

# Rebuild the MusicVAE model with batch_size=1
def build_vae(vocab_size, embedding_dim, latent_dim, seq_length):
    model = MusicVAE(vocab_size, embedding_dim, latent_dim, seq_length)
    # Call the model once with a dummy input to build its layers
    _ = model(tf.zeros((1, seq_length), dtype=tf.int32))
    return model

# Restore the model with the correct architecture
seq_length = 100 # Ensure this matches training seq_length
vocab_size = len(vocab) # Ensure this matches training vocab_size
embedding_dim = 256 # Ensure this matches training embedding_dim
latent_dim = 64 # Ensure this matches training latent_dim

vae = build_vae(vocab_size, embedding_dim, latent_dim, seq_length)

# Verify checkpoint directory and print available files
print("Checking for checkpoints in:", checkpoint_dir)
if os.path.exists(checkpoint_dir):
    checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.startswith('fast_cpu_vae')] # Filter for relevant checkpoints
    if checkpoint_files:
        # Sort to get the latest checkpoint by iteration number
        checkpoint_files.sort(key=lambda f: int(re.search(r'vae-(\d+)', f).group(1)) if re.search(r'vae-(\d+)', f) else 0)
        latest_checkpoint_path = os.path.join(checkpoint_dir, checkpoint_files[-1]) # Get the latest checkpoint path

        if latest_checkpoint_path:
            print("Found latest checkpoint file:", latest_checkpoint_path)
            try:
                vae.load_weights(latest_checkpoint_path)
                print("VAE model weights restored successfully.")
                vae.summary()
            except Exception as e:
                print(f"Error loading weights: {e}")
        else:
            print("No checkpoint files found in the directory.")
    else:
        print("No checkpoint files found in the directory.")
else:
    print("Checkpoint directory not found.")


In [ ]:
import tensorflow as tf
import numpy as np
import random
import re
from tqdm import tqdm
from IPython.display import Audio, display

# ============================================================
#              MusicVAE MODEL (simplified & optimized)
# ============================================================
class MusicVAE(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, latent_dim, seq_length):
        super(MusicVAE, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.latent_dim = latent_dim
        self.seq_length = seq_length

        # ===== Simplified Encoder =====
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        # Use cuDNN-compatible GRU settings for speed on GPU
        self.encoder_gru = tf.keras.layers.GRU(
            64,
            return_sequences=False,
            reset_after=True,   # IMPORTANT for cuDNN
        )
        self.encoder_dense = tf.keras.layers.Dense(128, activation='relu')
        self.encoder_dropout = tf.keras.layers.Dropout(0.2)
        self.z_mean_layer = tf.keras.layers.Dense(latent_dim)
        self.z_log_var_layer = tf.keras.layers.Dense(latent_dim)

        # ===== Simplified Decoder =====
        self.decoder_dense1 = tf.keras.layers.Dense(128, activation='relu')
        self.decoder_dense2 = tf.keras.layers.Dense(seq_length * 32, activation='relu')
        self.decoder_reshape = tf.keras.layers.Reshape((seq_length, 32))
        self.decoder_gru = tf.keras.layers.GRU(
            64,
            return_sequences=True,
            reset_after=True,   # IMPORTANT for cuDNN
        )
        self.output_dense = tf.keras.layers.Dense(vocab_size, activation='softmax')

    @staticmethod
    def sampling(args):
        """Reparameterization trick: z = μ + σ * ε"""
        z_mean, z_log_var = args
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    def encode(self, x):
        x = self.embedding(x)
        x = self.encoder_gru(x)
        x = self.encoder_dense(x)
        x = self.encoder_dropout(x)
        z_mean = self.z_mean_layer(x)
        z_log_var = self.z_log_var_layer(x)
        return z_mean, z_log_var

    def decode(self, z):
        x = self.decoder_dense1(z)
        x = self.decoder_dense2(x)
        x = self.decoder_reshape(x)
        x = self.decoder_gru(x)
        return self.output_dense(x)

    def call(self, inputs, training=None):
        z_mean, z_log_var = self.encode(inputs)
        z = self.sampling([z_mean, z_log_var])
        reconstructed = self.decode(z)
        return reconstructed, z_mean, z_log_var

    def generate(self, num_samples=1, temperature=1.0):
        z = tf.random.normal(shape=(num_samples, self.latent_dim))
        generated = self.decode(z)
        if temperature != 1.0:
            generated = generated / temperature
            generated = tf.nn.softmax(generated, axis=-1)
        return generated


# ============================================================
#                  Helper: sampling from probs
# ============================================================
def sample_from_probs(probs):
    """
    probs: 1D numpy array of shape (vocab_size,)
    Returns: sampled index according to the probability distribution.
    """
    probs = np.asarray(probs, dtype=np.float64)
    probs = probs / probs.sum()  # normalize to be safe
    return np.random.choice(len(probs), p=probs)


# ============================================================
#         Generate long song by chaining VAE segments
# ============================================================
def generate_long_song_by_chaining(
    vae_model,
    char2idx,
    idx2char,
    target_length=2000,
    seq_length=100,
    temperature=1.0,
):
    """
    Generate a long ABC string by repeatedly calling vae_model.generate()
    and concatenating segments until target_length characters are reached.

    To avoid tiny songs, we enforce a minimum generated length.
    """
    # Enforce a bigger minimum length so songs are clearly not "small"
    min_length = max(1600, 4 * seq_length)  # at least ~1600 chars
    target_length = max(target_length, min_length)

    generated_chars = []

    while len(generated_chars) < target_length:
        # One segment from VAE: (1, seq_length, vocab_size)
        segment_probs = vae_model.generate(num_samples=1, temperature=temperature)[0].numpy()

        # Sampling instead of argmax -> more notes, more variation
        segment_indices = [sample_from_probs(p) for p in segment_probs]
        segment_chars = [idx2char[idx] for idx in segment_indices]
        generated_chars.extend(segment_chars)

    return ''.join(generated_chars[:target_length])


# ============================================================
#             Basic ABC cleaning helpers / stubs
# ============================================================
def clean_abc_segment(segment):
    """
    Simple cleaner: keep printable characters, strip weird control chars.
    You can replace this with your own more advanced cleaner.
    """
    # Keep common ABC chars plus whitespace
    allowed = set("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789/|:[]!#=_+-,().' \"\n\r\t")
    return ''.join(c for c in segment if c in allowed)


def fix_abc_for_audio(abc_string):
    """
    Ensure the ABC has a basic header. If there's already a header starting
    with X:, we keep it. Otherwise, prepend a default header.
    """
    if abc_string.strip().startswith("X:"):
        cleaned = abc_string
    else:
        header = "X:1\nT:Generated\nM:4/4\nL:1/8\nK:C\n"
        cleaned = header + abc_string

    # Clean body a bit
    cleaned = clean_abc_segment(cleaned)
    # Ensure it ends with a newline
    if not cleaned.endswith("\n"):
        cleaned += "\n"
    return cleaned


# ============================================================
#   Convert ABC to audio by synthesizing simple sine waves
# ============================================================
def convert_abc_to_audio(abc_string, filename_prefix="song",
                         note_duration=0.08, sr=16000):
    """
    Very simple synthesizer:
    - Extract letters A–G / a–g as notes
    - Map them to basic sine tones
    - Each note gets fixed duration (note_duration seconds)
    - Return an IPython.display.Audio object

    This is not a true ABC parser, but it gives you audio whose
    length scales with the note count.
    """
    # Extract notes
    notes = re.findall(r'[A-Ga-g]', abc_string)
    if not notes:
        print(f"[WARN] No notes found in ABC for {filename_prefix}.")
        return None

    # Basic frequency map for one octave
    base_freqs = {
        'C': 261.63,
        'D': 293.66,
        'E': 329.63,
        'F': 349.23,
        'G': 392.00,
        'A': 440.00,
        'B': 493.88,
    }

    all_samples = []

    t = np.linspace(0, note_duration, int(sr * note_duration), endpoint=False)

    for n in notes:
        name = n.upper()
        freq = base_freqs.get(name, 440.0)  # default A if unknown
        if n.islower():
            freq *= 2.0  # lowercase = one octave up

        wave = 0.3 * np.sin(2 * np.pi * freq * t)
        all_samples.append(wave)

    audio_samples = np.concatenate(all_samples).astype(np.float32)

    # Optional tiny fade-out to avoid click at end
    if len(audio_samples) > sr // 50:
        fade_len = sr // 100
        fade = np.linspace(1.0, 0.0, fade_len)
        audio_samples[-fade_len:] *= fade

    return Audio(audio_samples, rate=sr)


# ============================================================
#         Function to Reconstruct the Original Song
# ============================================================
def reconstruct_original_song(vae_model, original_song, char2idx, idx2char, seq_length=64):
    """Attempt to reconstruct the original song using VAE encode-decode"""
    print(" Reconstructing the original song through VAE...")

    # Convert original song to indices
    song_indices = [char2idx.get(c, 0) for c in original_song]
    original_length = len(song_indices)

    # Process the entire song in overlapping segments
    reconstructed_parts = []
    overlap = 16  # Overlap for continuity

    for i in tqdm(range(0, original_length, seq_length - overlap), desc="Reconstructing segments"):
        # Get current segment
        segment = song_indices[i:i+seq_length]
        if len(segment) < seq_length:
            segment += [0] * (seq_length - len(segment))  # Pad if needed

        # Encode the original segment to latent space
        input_seq = tf.expand_dims(segment, 0)
        z_mean, z_log_var = vae_model.encode(input_seq)

        # Use mean (no sampling) for more accurate reconstruction
        z = z_mean

        # Decode back to music
        reconstructed_logits = vae_model.decode(z)
        reconstructed_indices = tf.argmax(reconstructed_logits[0], axis=-1).numpy()

        # Convert to characters
        if i == 0:
            # First segment - take all
            segment_chars = [idx2char[idx] for idx in reconstructed_indices]
            reconstructed_parts.append(''.join(segment_chars))
        else:
            # Subsequent segments - skip overlap
            segment_chars = [idx2char[idx] for idx in reconstructed_indices[overlap:]]
            reconstructed_parts.append(''.join(segment_chars))

    result = ''.join(reconstructed_parts)

    # Trim to original length
    if len(result) > original_length:
        result = result[:original_length]

    print(f"Reconstructed {len(result)} characters (original was {original_length})")
    return result


# ============================================================
#         Alternative: Iterative Reconstruction (Optional)
# ============================================================
def reconstruct_iteratively(vae_model, original_song, char2idx, idx2char, seq_length=64):
    """Reconstruct by iteratively processing the original song"""
    print(" Iterative reconstruction of original song...")

    # Extract header from original
    header_match = re.search(r'(X:.*?K:[^\n]+\n)', original_song, re.DOTALL)
    if header_match:
        header = header_match.group(1)
        body = original_song[len(header):]
    else:
        header = "X:1\nT:Reconstructed\nM:4/4\nL:1/8\nK:C\n"
        body = original_song

    # Start with original header
    result = header

    # Process original body in chunks to guide reconstruction
    body_indices = [char2idx.get(c, 0) for c in body]

    for i in tqdm(range(0, len(body_indices), seq_length//2), desc="Iterative reconstruction"):
        # Take a chunk from original as context
        context_chunk = body_indices[i:i+seq_length//2]
        if len(context_chunk) < seq_length//2:
            context_chunk += [0] * (seq_length//2 - len(context_chunk))

        # Pad to full sequence length
        full_context = (result[-seq_length//2:] if len(result) >= seq_length//2 else result)
        context_indices = [char2idx.get(c, 0) for c in full_context] + context_chunk

        if len(context_indices) < seq_length:
            context_indices += [0] * (seq_length - len(context_indices))
        context_indices = context_indices[-seq_length:]  # Take last seq_length

        # Encode and decode
        input_seq = tf.expand_dims(context_indices, 0)
        z_mean, z_log_var = vae_model.encode(input_seq)

        # Add slight variation
        epsilon = tf.random.normal(shape=tf.shape(z_mean)) * 0.1
        z = z_mean + epsilon

        # Decode
        output = vae_model.decode(z)
        decoded_indices = tf.argmax(output[0], axis=-1).numpy()

        # Take latter half as the "predicted" continuation
        new_part = decoded_indices[seq_length//2:]
        new_chars = ''.join([idx2char[idx] for idx in new_part])

        # Clean and add
        cleaned_part = clean_abc_segment(new_chars)
        result += cleaned_part[:32]  # Add 32 chars at a time

    print(f"Iteratively reconstructed {len(result)} characters")
    return result


# ============================================================
#                    MAIN (same hyperparameters)
# ============================================================
def main():
    print(" Loading dataset and VAE model...")
    import mitdeeplearning as mdl
    songs = mdl.lab1.load_training_data()

    vocab = sorted(set("".join(songs)))
    char2idx = {u: i for i, u in enumerate(vocab)}
    idx2char = np.array(vocab)

    # Load VAE model
    try:
        seq_length = 100
        vocab_size = len(vocab)
        embedding_dim = 256
        latent_dim = 64

        vae_model = MusicVAE(vocab_size, embedding_dim, latent_dim, seq_length)
        dummy_input = tf.zeros((1, seq_length), dtype=tf.int32)
        _ = vae_model(dummy_input)
        vae_model.load_weights("./training_checkpoints/fast_cpu_vae.weights.h5")
        print(" VAE model loaded successfully.")

    except Exception as e:
        print(f" Failed to load VAE model: {e}")
        return

    # Pick original song
    original_song = random.choice(songs)
    original_length = len(original_song)

    print(f"\n Selected original song ({original_length} characters):")
    print(original_song[:200] + "...")

    # 1. GENERATED: Create a completely new song using VAE
    print(f"\n Creating GENERATED song (new composition)...")
    generated_song = generate_long_song_by_chaining(
        vae_model, char2idx, idx2char, target_length=original_length * 2, seq_length=seq_length
    )

    # 2. RECONSTRUCTED: Attempt to recreate the selected original song
    print(f"\n Creating RECONSTRUCTED song (recreating selected song)...")
    reconstructed_song = reconstruct_original_song(
        vae_model, original_song, char2idx, idx2char, seq_length
    )

    # Fix both for audio conversion
    generated_song = fix_abc_for_audio(generated_song)
    reconstructed_song = fix_abc_for_audio(reconstructed_song)

    # Save files with clear names
    with open("1_original_song.abc", "w") as f:
        f.write(original_song)
    with open("2_generated_song.abc", "w") as f:
        f.write(generated_song)
    with open("3_reconstructed_song.abc", "w") as f:
        f.write(reconstructed_song)

    # Length comparison
    print(f"\n Song comparison:")
    print(f"ORIGINAL:      {len(original_song):4d} chars")
    print(f"GENERATED:     {len(generated_song):4d} chars")
    print(f"RECONSTRUCTED: {len(reconstructed_song):4d} chars")

    # Count notes
    orig_notes = len(re.findall(r'[A-Ga-g]', original_song))
    gen_notes = len(re.findall(r'[A-Ga-g]', generated_song))
    rec_notes = len(re.findall(r'[A-Ga-g]', reconstructed_song))

    print(f"ORIGINAL notes:      {orig_notes}")
    print(f"GENERATED notes:     {gen_notes}")
    print(f"RECONSTRUCTED notes: {rec_notes}")

    # Convert to audio (save WAVs)
    print("\n Converting to audio and saving WAV files...")
    original_audio = convert_abc_to_audio(original_song, "1_original_song")
    generated_audio = convert_abc_to_audio(generated_song, "2_generated_song")
    reconstructed_audio = convert_abc_to_audio(reconstructed_song, "3_reconstructed_song")

    # Play audio with clear descriptions
    print("\n🎧 AUDIO COMPARISON:")

    if original_audio:
        print(f"\n 1. ORIGINAL (the selected song):")
        display(original_audio)

    if generated_audio:
        print(f"\n 2. GENERATED (new VAE composition):")
        display(generated_audio)

    if reconstructed_audio:
        print(f"\n 3. RECONSTRUCTED (VAE trying to recreate the original):")
        display(reconstructed_audio)

    print("\n Compare how well the VAE reconstructed vs. generated new music!")


if __name__ == "__main__":
    main()

## 1.8 Evaluation: Comparing Generated and Original Songs
### Metrics: Jaccard, Pitch Histogram, Repetition, n-gram, Contour,Coherence, BLEU Score, Sequence Similarity


In [ ]:
from music21 import converter, note
from collections import Counter
from difflib import SequenceMatcher
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np
import re

# --- Enhanced ABC fixing function ---
def fix_abc_notation_enhanced(abc_text):
    lines = abc_text.split('\n')
    fixed_lines = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Fix specific malformed headers
        if line.startswith('M::'):
            line = 'M:4/4'  # Default meter
        elif line.startswith('M:') and line == 'M:':
            line = 'M:4/4'
        elif line.startswith('L::'):
            line = 'L:1/8'  # Default note length
        elif line.startswith('L:') and line == 'L:':
            line = 'L:1/8'
        elif line.startswith('K::'):
            line = 'K:C'    # Default key
        elif line.startswith('K:') and line == 'K:':
            line = 'K:C'
        elif line.startswith('T::'):
            line = 'T:Generated Song'
        elif line.startswith('T:') and line == 'T:':
            line = 'T:Generated Song'
        elif line.startswith('X::'):
            line = 'X:1'
        elif line.startswith('X:') and line == 'X:':
            line = 'X:1'
        # Handle empty headers with just colon
        elif ':' in line and line.endswith(':'):
            header = line.split(':')[0]
            if header == 'M':
                line = 'M:4/4'
            elif header == 'L':
                line = 'L:1/8'
            elif header == 'K':
                line = 'K:C'
            elif header == 'T':
                line = 'T:Generated Song'
            elif header == 'X':
                line = 'X:1'

        fixed_lines.append(line)

    return '\n'.join(fixed_lines)

# --- Updated parsing function ---
def parse_abc_notes_fixed(abc_text, filename=""):
    print(f"\n=== PARSING ABC: {filename} ===")

    if not abc_text.strip():
        print("Empty ABC content")
        return []

    # Try direct parsing first
    try:
        print("Attempt 1: Direct music21 parsing...")
        parsed = converter.parse(abc_text, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f"Success! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f"Failed: {e}")

    # Apply enhanced fixes and retry
    try:
        print("Attempt 2: Enhanced ABC fixing...")
        fixed_abc = fix_abc_notation_enhanced(abc_text)

        # Show what was fixed
        if fixed_abc != abc_text:
            print("Applied fixes:")
            original_lines = abc_text.split('\n')
            fixed_lines = fixed_abc.split('\n')
            for i, (orig, fixed) in enumerate(zip(original_lines, fixed_lines)):
                if orig != fixed:
                    print(f"  Line {i+1}: '{orig}' → '{fixed}'")

        parsed = converter.parse(fixed_abc, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f"Success after enhanced fixes! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f"Enhanced fixing failed: {e}")

    # Fallback to manual extraction
    try:
        print("Attempt 3: Manual note extraction...")
        notes = extract_notes_manually(abc_text)
        if notes:
            print(f" Manual extraction found {len(notes)} notes")
            return notes
    except Exception as e:
        print(f"Manual extraction failed: {e}")

    print("All parsing attempts failed")
    return []

# --- Manual note extraction (improved) ---
def extract_notes_manually(abc_text):
    lines = abc_text.split('\n')
    note_lines = []

    # Skip header lines and collect note content
    for line in lines:
        line = line.strip()
        # Skip empty lines and headers
        if not line or line.startswith(('X:', 'T:', 'M:', 'L:', 'K:', 'Q:', 'Z:', '%')):
            continue
        note_lines.append(line)

    if not note_lines:
        return []

    # Combine all note content and clean it
    note_content = ' '.join(note_lines)

    # Remove bar lines, repeat marks, and other symbols
    note_content = re.sub(r'[|\[\]!:]', ' ', note_content)

    # Extract notes with regex (handles various ABC notation)
    note_pattern = r'[A-Ga-g][#b]?[,\']*[0-9]*'
    matches = re.findall(note_pattern, note_content)

    # Convert to standard note names
    notes = []
    for match in matches:
        # Remove duration numbers
        clean_note = re.sub(r'[0-9]', '', match)
        if clean_note:  # Only process non-empty notes
            try:
                n = note.Note(clean_note)
                notes.append(n.nameWithOctave)
            except:
                continue

    return notes

# --- Keep all your original evaluation functions ---
def jaccard_overlap(a, b):
    set_a, set_b = set(a), set(b)
    return len(set_a & set_b) / len(set_a | set_b) if (set_a | set_b) else 0

def pitch_class_histogram(notes):
    h = {p: 0 for p in "CDEFGAB"}
    for n in notes:
        if n[0] in h:
            h[n[0]] += 1
    total = sum(h.values()) or 1
    return {k: v / total for k, v in h.items()}

def histogram_difference(h1, h2):
    return sum(abs(h1[k] - h2[k]) for k in h1)

def repetition_factor(notes):
    return sum(notes.count(n) for n in set(notes)) / len(notes) if notes else 0

def ngram_overlap(a, b, n=2):
    def extract_ngrams(seq, n):
        return Counter(tuple(seq[i:i+n]) for i in range(len(seq) - n + 1))
    a_ngrams = extract_ngrams(a, n)
    b_ngrams = extract_ngrams(b, n)
    shared = sum((a_ngrams & b_ngrams).values())
    total = sum((a_ngrams | b_ngrams).values())
    return shared / total if total else 0

def melodic_contour(notes):
    def to_midi(n):
        try: return note.Note(n).pitch.midi
        except: return None
    mids = [to_midi(n) for n in notes if n]
    mids = [m for m in mids if m is not None]
    contour = []
    for i in range(1, len(mids)):
        if mids[i] > mids[i-1]: contour.append('U')
        elif mids[i] < mids[i-1]: contour.append('D')
        else: contour.append('S')
    return contour

def contour_similarity(c1, c2):
    return SequenceMatcher(None, c1, c2).ratio()

def musical_coherence(notes):
    def to_midi(n):
        try: return note.Note(n).pitch.midi
        except: return None
    mids = [to_midi(n) for n in notes if n]
    mids = [m for m in mids if m is not None]
    if len(mids) < 2: return 1.0
    intervals = [mids[i+1] - mids[i] for i in range(len(mids) - 1)]
    std_dev = np.std(intervals)
    max_possible = 87
    return 1.0 - min(1.0, std_dev / max_possible)

def compute_bleu(reference, candidate):
    try:
        smoothing = SmoothingFunction().method1
        return sentence_bleu([reference], candidate, smoothing_function=smoothing)
    except Exception:
        return 0.0

def sequence_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def evaluate_similarity(original_notes, generated_notes):
    if not original_notes or not generated_notes:
        print("One or both note sequences are empty. Check ABC content.")
        return

    print("\n MUSIC SIMILARITY EVALUATION")
    print("="*50)
    print(f" Original: {len(original_notes)} notes | Generated: {len(generated_notes)} notes")
    print(f"\n Similarity Metrics:")
    print(f"   Note Overlap (Jaccard):        {jaccard_overlap(original_notes, generated_notes):.3f}")

    h_orig = pitch_class_histogram(original_notes)
    h_gen = pitch_class_histogram(generated_notes)
    print(f"   Pitch Class Histogram Δ:       {histogram_difference(h_orig, h_gen):.3f}")
    print(f"   Repetition Factor (original):  {repetition_factor(original_notes):.3f}")
    print(f"   Repetition Factor (generated): {repetition_factor(generated_notes):.3f}")
    print(f"   Bigram Overlap:                {ngram_overlap(original_notes, generated_notes, 2):.3f}")
    print(f"   Trigram Overlap:               {ngram_overlap(original_notes, generated_notes, 3):.3f}")

    contour_orig = melodic_contour(original_notes)
    contour_gen = melodic_contour(generated_notes)
    print(f"   Melodic Contour Similarity:    {contour_similarity(contour_orig, contour_gen):.3f}")
    print(f"   Musical Coherence (original):  {musical_coherence(original_notes):.3f}")
    print(f"   Musical Coherence (generated): {musical_coherence(generated_notes):.3f}")
    print(f"   BLEU Score:                    {compute_bleu(original_notes, generated_notes):.3f}")
    print(f"   Sequence Similarity:           {sequence_similarity(original_notes, generated_notes):.3f}")

# --- Load ABC from file ---
def load_abc_from_file(filepath):
    try:
        with open(filepath, 'r') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return ""

# --- Main execution ---
original_song = load_abc_from_file("/content/1_original_song.abc")
reconstructed_song = load_abc_from_file("/content/3_reconstructed_song.abc")

# Use the enhanced parsing function
original_notes = parse_abc_notes_fixed(original_song, "original")
reconstructed_notes = parse_abc_notes_fixed(reconstructed_song, "reconstructed")

evaluate_similarity(original_notes, reconstructed_notes)

#Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from music21 import converter, note
from collections import Counter
import re

# --- Enhanced ABC fixing function ---
def fix_abc_notation_enhanced(abc_text):
    lines = abc_text.split('\n')
    fixed_lines = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Fix specific malformed headers
        if line.startswith('M::'):
            line = 'M:4/4'
        elif line.startswith('M:') and line == 'M:':
            line = 'M:4/4'
        elif line.startswith('L::'):
            line = 'L:1/8'
        elif line.startswith('L:') and line == 'L:':
            line = 'L:1/8'
        elif line.startswith('K::'):
            line = 'K:C'
        elif line.startswith('K:') and line == 'K:':
            line = 'K:C'
        elif line.startswith('T::'):
            line = 'T:Generated Song'
        elif line.startswith('T:') and line == 'T:':
            line = 'T:Generated Song'
        elif line.startswith('X::'):
            line = 'X:1'
        elif line.startswith('X:') and line == 'X:':
            line = 'X:1'
        elif ':' in line and line.endswith(':'):
            header = line.split(':')[0]
            if header == 'M':
                line = 'M:4/4'
            elif header == 'L':
                line = 'L:1/8'
            elif header == 'K':
                line = 'K:C'
            elif header == 'T':
                line = 'T:Generated Song'
            elif header == 'X':
                line = 'X:1'

        fixed_lines.append(line)

    return '\n'.join(fixed_lines)

# --- Load ABC from file ---
def load_abc_from_file(filepath):
    try:
        with open(filepath, 'r') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return ""

# --- Enhanced ABC parsing with error handling ---
def parse_abc_notes_fixed(abc_text, filename=""):
    print(f"\n=== PARSING ABC: {filename} ===")

    if not abc_text.strip():
        print(" Empty ABC content")
        return []

    # Try direct parsing first
    try:
        print("Attempt 1: Direct music21 parsing...")
        parsed = converter.parse(abc_text, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f"Success! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f"Failed: {e}")

    # Apply enhanced fixes and retry
    try:
        print("Attempt 2: Enhanced ABC fixing...")
        fixed_abc = fix_abc_notation_enhanced(abc_text)

        # Show what was fixed
        if fixed_abc != abc_text:
            print("Applied fixes:")
            original_lines = abc_text.split('\n')
            fixed_lines = fixed_abc.split('\n')
            for i, (orig, fixed) in enumerate(zip(original_lines, fixed_lines)):
                if orig != fixed:
                    print(f"  Line {i+1}: '{orig}' → '{fixed}'")

        parsed = converter.parse(fixed_abc, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f" Success after enhanced fixes! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f" Enhanced fixing failed: {e}")

    print(" All parsing attempts failed")
    return []

# --- Align sequences for fair comparison ---
def align_sequences(seq1, seq2):
    """Align sequences to same length for meaningful comparison"""
    min_length = min(len(seq1), len(seq2))
    return seq1[:min_length], seq2[:min_length]

# --- Create unified vocabulary ---
def create_unified_vocabulary(seq1, seq2):
    """Create unified vocabulary from both sequences"""
    all_notes = set(seq1 + seq2)
    vocab = sorted(list(all_notes))
    note_to_id = {note: i for i, note in enumerate(vocab)}
    return vocab, note_to_id

# --- Enhanced Confusion Matrix Plot ---
def plot_confusion_matrix(y_true_notes, y_pred_notes, title="Confusion Matrix"):
    """Enhanced confusion matrix that handles different vocabularies and sequence lengths"""

    if not y_true_notes or not y_pred_notes:
        print(" Cannot create confusion matrix: empty sequences")
        return

    print(f"\n Creating confusion matrix...")
    print(f"   Original: {len(y_true_notes)} notes")
    print(f"   Generated: {len(y_pred_notes)} notes")

    # Align sequences to same length
    y_true_aligned, y_pred_aligned = align_sequences(y_true_notes, y_pred_notes)
    print(f"   Aligned length: {len(y_true_aligned)}")

    if len(y_true_aligned) == 0:
        print(" No notes to compare after alignment")
        return

    # Create unified vocabulary
    vocab, note_to_id = create_unified_vocabulary(y_true_aligned, y_pred_aligned)

    # Convert to IDs
    y_true_ids = [note_to_id[note] for note in y_true_aligned]
    y_pred_ids = [note_to_id[note] for note in y_pred_aligned]

    # Create confusion matrix
    cm = confusion_matrix(y_true_ids, y_pred_ids, labels=range(len(vocab)))

    # Calculate accuracy
    accuracy = sum(1 for t, p in zip(y_true_ids, y_pred_ids) if t == p) / len(y_true_ids)

    # Create the plot
    plt.figure(figsize=(max(8, len(vocab) * 0.6), max(6, len(vocab) * 0.5)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=vocab, yticklabels=vocab)
    plt.xlabel("Predicted Notes")
    plt.ylabel("True Notes")
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# --- Example Usage ---
if __name__ == "__main__":
    print(" ABC Confusion Matrix Analysis")
    print("="*50)

    # Load the files
    original_song = load_abc_from_file("/content/1_original_song.abc")
    reconstructed_song = load_abc_from_file("/content/3_reconstructed_song.abc")

    # Parse using the correct function name
    original_notes = parse_abc_notes_fixed(original_song, "original")
    reconstructed_notes = parse_abc_notes_fixed(reconstructed_song, "reconstructed")

    # Create confusion matrix
    if original_notes and reconstructed_notes:
        plot_confusion_matrix(original_notes, reconstructed_notes,
                             "Original vs Reconstructed Notes")
    else:
        print(" Could not create confusion matrix: parsing failed")

#Audio Similarity(DTW)

In [ ]:
# audio_similarity_eval.py
import librosa
import numpy as np
from librosa.sequence import dtw

# --- Audio Similarity using DTW over MFCCs ---
def compute_audio_similarity_dtw(file1, file2, sr=22050, n_mfcc=20):
    try:
        # Load audio
        y1, _ = librosa.load(file1, sr=sr)
        y2, _ = librosa.load(file2, sr=sr)

        # Extract MFCCs (shape: n_mfcc × time)
        mfcc1 = librosa.feature.mfcc(y=y1, sr=sr, n_mfcc=n_mfcc)
        mfcc2 = librosa.feature.mfcc(y=y2, sr=sr, n_mfcc=n_mfcc)

        # Compute DTW (match MFCC features over time using cosine distance)
        D, wp = dtw(X=mfcc1, Y=mfcc2, metric='cosine')

        # Normalize distance by length of path
        dist = D[-1, -1] / len(wp)
        similarity = 1 / (1 + dist)  # Higher is better
        print(f"DTW Audio Similarity: {similarity:.2f}")
        return similarity

    except Exception as e:
        print(f"Error during DTW audio comparison: {e}")
        return 0.0

# --- Example Usage ---
if __name__ == "__main__":
    file1 = "/content/1_original_song.wav"
    file2 = "/content/3_reconstructed_song.wav"
    compute_audio_similarity_dtw(file1, file2)

#Music Spectogram

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

# Load the original audio file
original_audio_path = "/content/1_original_song.wav"  # Or "original_song.mp3"
y_orig, sr_orig = librosa.load(original_audio_path, sr=None)

# Create and display spectrogram for original music
plt.figure(figsize=(10, 4))
S_orig = librosa.feature.melspectrogram(y=y_orig, sr=sr_orig)
librosa.display.specshow(librosa.power_to_db(S_orig, ref=np.max),
                         sr=sr_orig, x_axis='time', y_axis='mel')
plt.title("Original Music Spectrogram")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

# Load generated audio file
generated_audio_path = "/content/3_reconstructed_song.wav"  # or .wav
y_gen, sr_gen = librosa.load(generated_audio_path, sr=None)

# Create and display spectrogram for generated music
plt.figure(figsize=(10, 4))
S_gen = librosa.feature.melspectrogram(y=y_gen, sr=sr_gen)
librosa.display.specshow(librosa.power_to_db(S_gen, ref=np.max),
                         sr=sr_gen, x_axis='time', y_axis='mel')
plt.title("Generated Music Spectrogram")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

## Generating Multiple songs

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from IPython.display import Audio, display
from tqdm import tqdm
import re

# Fix ABC rhythm
def fix_abc_rhythm(abc_string):
    header_match = re.search(r'(X:.*?K:[^\n]+\n)', abc_string, re.DOTALL)
    header = header_match.group(1) if header_match else "X:1\nT:Generated\nM:4/4\nL:1/8\nK:C\n"
    if "Q:" not in header:
        header += "Q:1/4=30\n"
    body = abc_string.replace(header, "")
    body = re.sub(r'[zZxX|:\[\]\s]+', '', body)
    notes = re.findall(r"[A-Ga-g][,']*", body)
    durations = ["1/8", "1/8", "1/8", "1/8", "1/4", "1/4", "1/8", "1/8"]
    cleaned_notes = [note + durations[i % len(durations)] for i, note in enumerate(notes)]
    measures = [" ".join(cleaned_notes[i:i+8]) for i in range(0, len(cleaned_notes), 8)]
    return header.strip() + "\n" + " |\n".join(measures) + " |"

# Convert ABC to audio
def convert_abc_to_audio(abc_text, filename):
    try:
        abc_path = f"{filename}.abc"
        with open(abc_path, "w") as f:
            f.write(abc_text)
        midi_path = f"{filename}.mid"
        os.system(f"abc2midi {abc_path} -o {midi_path} > /dev/null 2>&1")
        wav_path = f"{filename}.wav"
        os.system(f"timidity {midi_path} -Ow -o {wav_path} > /dev/null 2>&1")
        print(f" Audio saved: {wav_path}")
        return wav_path
    except Exception as e:
        print(f" Error converting {filename}: {e}")
        return None

# Generate song from decoder
def generate_song_from_model(decoder, latent_dim, idx2char, repeat=15, temperature=0.5):
    generated_song = []
    for _ in tqdm(range(repeat), desc="Generating"):
        z = tf.random.normal((1, latent_dim)) * temperature
        logits = decoder(z)
        indices = tf.argmax(logits[0], axis=-1).numpy()
        chars = [idx2char[i] for i in indices]
        generated_song.extend(chars)

    raw_output = ''.join(generated_song)
    cleaned_output = fix_abc_rhythm(raw_output)
    return cleaned_output

# Main function
def main(num_songs=3):
    print("Loading dataset and decoder model...")
    import mitdeeplearning as mdl
    songs = mdl.lab1.load_training_data()

    vocab = sorted(set("".join(songs)))
    char2idx = {u: i for i, u in enumerate(vocab)}
    idx2char = np.array(vocab)

    # Parameters
    seq_length = 100
    vocab_size = len(vocab)
    latent_dim = 32

    # Define decoder
    class VAE_Decoder(tf.keras.Model):
        def __init__(self, seq_length, vocab_size, latent_dim):
            super().__init__()
            self.d1 = tf.keras.layers.Dense(64, activation='relu')
            self.d2 = tf.keras.layers.Dense(seq_length * vocab_size, activation='sigmoid')
            self.reshape = tf.keras.layers.Reshape((seq_length, vocab_size))

        def call(self, z):
            x = self.d1(z)
            x = self.d2(x)
            return self.reshape(x)

    decoder = VAE_Decoder(seq_length, vocab_size, latent_dim)
    try:
        decoder.build((1, latent_dim))
        decoder.load_weights("/content/training_checkpoints/fast_cpu_vae.weights.h5")
        print("Decoder weights loaded successfully")
    except Exception as e:
        print(f"Failed to load model: {e}")
        return

    for i in range(num_songs):
        print(f"\n==================== Pair {i+1} ====================")
        original_song = random.choice(songs)

        generated_song = generate_song_from_model(
            decoder, latent_dim, idx2char,
            repeat=15,  # 15 x 100 chars
            temperature=0.5
        )

        with open(f"original_song_{i+1}.abc", "w") as f:
            f.write(original_song)

        with open(f"generated_song_{i+1}.abc", "w") as f:
            f.write(generated_song)

        original_audio = convert_abc_to_audio(original_song, f"original_song_{i+1}")
        generated_audio = convert_abc_to_audio(generated_song, f"generated_song_{i+1}")

        if original_audio:
            print(f"\n Playing original song {i+1}:")
            display(Audio(filename=original_audio))

        if generated_audio:
            print(f"\n Playing generated song {i+1}:")
            display(Audio(filename=generated_audio))

#  Run
if __name__ == "__main__":
    main(num_songs=8)